In [3]:

# Install IBL ONE API
%pip -q install -U ONE-api

from one.api import ONE
import pandas as pd
import numpy as np

# Connect using the public IBL credentials
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    username="intbrainlab",
    password="international",
    silent=True
)

PROJECT = "angelaki_mouseASD"
RELEASE = "2025_Q3_Noel_et_al_Autism"

print("Connected to IBL OpenAlyx")
print("Project:", PROJECT)
print("Release:", RELEASE)


Connected to IBL OpenAlyx
Project: angelaki_mouseASD
Release: 2025_Q3_Noel_et_al_Autism


In [4]:

# Find all probe insertions belonging to the Angelaki ASD project
insertions = one.search_insertions(project=PROJECT)

print("Total probe insertions:", len(insertions))
print("First five insertion IDs:")
for pid in insertions[:5]:
    print(pid)

# Keep the insertion list for the next steps
pids = [str(pid) for pid in insertions]


/usr/local/lib/python3.13/dist-packages/one/webclient.py:1053: RuntimeWarning: The shared public account will be retired soon. Register your own account at https://openalyx.internationalbrainlab.org/signup
  emit(message)


Total probe insertions: 364
First five insertion IDs:
71275b79-12f3-42a0-8963-43f6381fbd4a
a24cad0d-9d17-472c-aac2-f5ee4a126bb5
8a05e4d5-a860-4947-81d5-9a2c90779d44
617f42c7-9580-4f25-a466-ec8989d89452
c4a5fed9-ff3b-4b73-8a79-38ba966c0093


In [5]:

# Select one example probe insertion
pid = pids[0]

# Read insertion metadata without downloading raw signals
insertion_info = one.alyx.rest(
    "insertions", "read", id=pid
)

print("Insertion ID:", pid)
print("Probe name:", insertion_info.get("name"))

# Resolve the associated session
session_ref = insertion_info.get("session")
eid = (
    session_ref.split("/")[-2]
    if isinstance(session_ref, str) and "/" in session_ref
    else str(session_ref)
)

print("Session ID:", eid)

# List available files (metadata only)
datasets = one.list_datasets(
    eid, details=True
)

df = pd.DataFrame(datasets)

print("Total dataset entries:", len(df))
print("Available columns:", df.columns.tolist())

display(df.head(10))


Insertion ID: 71275b79-12f3-42a0-8963-43f6381fbd4a
Probe name: probe00
Session ID: 3da15f76-b31b-4ca7-a25c-c46cb8e40a06
Total dataset entries: 127
Available columns: ['file_size', 'hash', 'exists', 'rel_path', 'default_revision', 'qc']


,file_size,hash,exists,rel_path,default_revision,qc
id,,,,,,
02c09856-5183-401e-977e-101b94694166,86640,292e611afefee0d24bc93e7e45b2800d,True,raw_ephys_data/probe01/_spikeglx_sync.polariti...,True,NOT_SET
043642a1-de53-4eb3-a913-1922461f6e93,13900456,11eb9ba41106ef32b55243dd52be1f63,True,alf/_ibl_wheel.timestamps.npy,True,NOT_SET
0598f25b-fa55-4b77-8e6d-487b0224ce31,86640,3ca33ded4d6b5fc4dcc2a7a17f69e2b1,True,raw_ephys_data/probe00/_spikeglx_sync.times.pr...,True,NOT_SET
0778952c-5693-420b-9b52-af168406aac6,37000368,d52bff48b96dddc235b022f6b96688f0,True,alf/probe01/pykilosort/spikes.samples.npy,True,NOT_SET
08b47448-9564-4958-b7d9-96974663d2c0,26434960,5c64643d163456c0bcd7e1ab643b6e02,True,alf/probe00/pykilosort/spikes.amps.npy,True,NOT_SET
0a3a82af-cf08-40ed-b075-1e7a9fd4084d,33386,0c232d2e5e6f5b7b1276b5d31ea77984,True,alf/probe00/pykilosort/#2024-03-12#/clusters.m...,False,NOT_SET
0f644cfd-ed29-4980-91d3-20e0a5647675,13806,f9ad7f0a697b84d3a20e5f09f5b1d43b,True,alf/probe01/pykilosort/clusters.uuids.csv,True,NOT_SET
11574cdc-14c4-40d5-b5f8-2ecc18e4443f,13217544,5841365092da533766f88116c32c3f7d,True,alf/probe00/pykilosort/spikes.clusters.npy,True,NOT_SET
12631122-7a61-468e-accd-de9fcae81041,3867330,83532f8396d224f141d602341f0012a0,True,alf/_ibl_bodyCamera.dlc.pqt,True,NOT_SET


In [6]:

# Search dataset metadata for continuous LF recordings
# and existing spectral summaries

df["dataset_path"] = df.get(
    "rel_path",
    pd.Series("", index=df.index)
).fillna("").astype(str)

lf_mask = df["dataset_path"].str.contains(
    r"\.lf\.|SpectralDensityLF|ephysTimeRmsLF",
    case=False,
    regex=True
)

lf_datasets = df.loc[lf_mask].copy()

print("Potential LFP-related files:", len(lf_datasets))
display(lf_datasets)


Potential LFP-related files: 14


,file_size,hash,exists,rel_path,default_revision,qc,dataset_path
id,,,,,,,
2c9239a9-5db3-42ba-b316-9c5435aeeb7a,2542668,1dc5dc4adfb7e13d5579aff7409701ec,True,raw_ephys_data/probe00/_iblqc_ephysTimeRmsLF.r...,True,NOT_SET,raw_ephys_data/probe00/_iblqc_ephysTimeRmsLF.r...
3bf46231-0b12-4f63-972c-80d017b9c0e6,2976804902,61db2b49b0a26b4c1025a1518b4ad34c,True,raw_ephys_data/probe00/_spikeglx_ephysData_g0_...,True,NOT_SET,raw_ephys_data/probe00/_spikeglx_ephysData_g0_...
728ac514-0788-4ff5-aacc-44c567adbc5e,2180,f3ffe62ed24d19c0979861b4308a9d00,True,raw_ephys_data/probe01/_iblqc_ephysSpectralDen...,True,NOT_SET,raw_ephys_data/probe01/_iblqc_ephysSpectralDen...
769884b3-f91f-4028-820e-8ee55c2e7ec5,6732,2dd9d05c37db730f0478295c292c0d4b,True,raw_ephys_data/probe01/_iblqc_ephysTimeRmsLF.t...,True,NOT_SET,raw_ephys_data/probe01/_iblqc_ephysTimeRmsLF.t...
81ecce23-d925-4772-b6ad-95cd8e9294f7,790148,70dfa3c0125be79645dcbadcec3484a7,True,raw_ephys_data/probe00/_iblqc_ephysSpectralDen...,True,NOT_SET,raw_ephys_data/probe00/_iblqc_ephysSpectralDen...
901d28ab-db2a-44bb-aa37-e97157ddffce,13904,09997db193dc51e60295f0e6f50f8e9e,True,raw_ephys_data/probe01/_spikeglx_ephysData_g0_...,True,NOT_SET,raw_ephys_data/probe01/_spikeglx_ephysData_g0_...
9e36dd4e-8ae3-4d9e-a3ee-f8fa60c16586,6732,aeaa70e86fbe700e57a30d8aa35e673f,True,raw_ephys_data/probe00/_iblqc_ephysTimeRmsLF.t...,True,NOT_SET,raw_ephys_data/probe00/_iblqc_ephysTimeRmsLF.t...
b72e2839-8d78-49e9-bc93-4e562a852dd7,2542668,fc53dcdae399ab3d70f3f19356456955,True,raw_ephys_data/probe01/_iblqc_ephysTimeRmsLF.r...,True,NOT_SET,raw_ephys_data/probe01/_iblqc_ephysTimeRmsLF.r...
c124fc1a-7678-42f5-b7fe-7bfaf84964e4,790148,ae958419415d176df564634a4175f321,True,raw_ephys_data/probe01/_iblqc_ephysSpectralDen...,True,NOT_SET,raw_ephys_data/probe01/_iblqc_ephysSpectralDen...
